# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


####  Run this cell to set up and start your interactive session.


In [5]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import Window, DataFrame, WindowSpec
import pyspark.sql.functions as F
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 60edfe01-3d41-49c7-a53d-4c7587a16683
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true


Following exception encountered while creating session: An error occurred (AlreadyExistsException) when calling the CreateSession operation: Session already created, sessionId=60edfe01-3d41-49c7-a53d-4c7587a16683 

Error message: Session already created, sessionId=60edfe01-3d41-49c7-a53d-4c7587a16683 

Traceback (most recent call last):
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/aws_glue_interactive_sessions_kernel/glue_kernel_utils/KernelGateway.py", line 104, in create_session
    response = self.glue_client.create_session(
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/client.py", line 602, in _api_call
    return self._make_api_call(operation_name, kwargs)
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/client.py", line 1078, in _make_api_call
    raise error_class(parsed_response,

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [6]:
def sma(df: DataFrame, n: int = 200) -> DataFrame:
    col_name: str = f"sma{n}"
    if col_name in df.columns:
        print(f"Skipping {col_name}: already exists.")
        return df
    window = Window.orderBy("datetime").rowsBetween(-n + 1, 0)
    return df.withColumn(col_name, F.avg("close").over(window))


def ema(df: DataFrame, n: int = 50) -> DataFrame:
    col_name: str = f"ema{n}"
    if col_name in df.columns:
        print(f"Skipping {col_name}: already exists.")
        return df

    alpha: float = 2 / (n + 1)
    window_unbounded = Window.orderBy("datetime").rowsBetween(Window.unboundedPreceding, 0)
    
    return df.withColumn(
        col_name,
        F.aggregate(
            F.collect_list("close").over(window_unbounded),
            F.lit(None).cast("double"),
            lambda acc, x: F.when(acc.isNull(), x).otherwise(acc + alpha * (x - acc))
        )
    )


def rsi(df: DataFrame, n: int = 14) -> DataFrame:
    col_name: str = f"rsi{n}"
    if col_name in df.columns:
        print(f"Skipping {col_name}: already exists.")
        return df

    window_prev = Window.orderBy("datetime")
    window_rolling = Window.orderBy("datetime").rowsBetween(-n + 1, 0)
    df = df.withColumn("diff", F.col("close") - F.lag("close", 1).over(window_prev))
    df = df.withColumn("gain", F.when(F.col("diff") > 0, F.col("diff")).otherwise(0))
    df = df.withColumn("loss", F.when(F.col("diff") < 0, F.abs(F.col("diff"))).otherwise(0))
    df = df.withColumn("avg_gain", F.avg("gain").over(window_rolling))
    df = df.withColumn("avg_loss", F.avg("loss").over(window_rolling))
    df = df.withColumn(
        "rs", 
        F.when(F.col("avg_loss") == 0, F.lit(None))
         .otherwise(F.col("avg_gain") / F.col("avg_loss"))
    )
    df_resultado = df.withColumn(
        col_name, 
        F.when(F.col("avg_loss") == 0, 100.0)
         .otherwise(100 - (100 / (1 + F.col("rs"))))
    )
    
    return df_resultado.drop("diff", "gain", "loss", "avg_gain", "avg_loss", "rs")


def macd(df: DataFrame, fast: int = 12, slow: int = 26, signal: int = 9) -> DataFrame:
    if "macd_line" in df.columns:
        print("Skipping MACD: already exists.")
        return df
    df = ema(df, n=fast)
    df = ema(df, n=slow)
    
    df = df.withColumn("macd_line", F.col(f"ema{fast}") - F.col(f"ema{slow}"))
    
    df_signal = df.withColumnRenamed("macd_line", "temp_col") \
                  .withColumnRenamed("close", "orig_close") \
                  .withColumnRenamed("temp_col", "close")
    
    df_signal = ema(df_signal, n=signal)
    
    df_resultado = df_signal.withColumnRenamed("close", "macd_line") \
                            .withColumnRenamed(f"ema{signal}", "signal_line") \
                            .withColumnRenamed("orig_close", "close") \
                            .withColumn("macd_histogram", F.col("macd_line") - F.col("signal_line"))
    
    return df_resultado.drop(f"ema{fast}", f"ema{slow}")

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 60edfe01-3d41-49c7-a53d-4c7587a16683
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true


Following exception encountered while creating session: An error occurred (AlreadyExistsException) when calling the CreateSession operation: Session already created, sessionId=60edfe01-3d41-49c7-a53d-4c7587a16683 

Error message: Session already created, sessionId=60edfe01-3d41-49c7-a53d-4c7587a16683 

Traceback (most recent call last):
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/aws_glue_interactive_sessions_kernel/glue_kernel_utils/KernelGateway.py", line 104, in create_session
    response = self.glue_client.create_session(
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/client.py", line 602, in _api_call
    return self._make_api_call(operation_name, kwargs)
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/client.py", line 1078, in _make_api_call
    raise error_class(parsed_response,

In [7]:
base_path: str = "s3://dodgy-consulting-trading-bucket-silver/DOGEUSD/"
df: DataFrame = spark.read.parquet(base_path)

df = sma(df, n=200)
df = ema(df, n=50)
df = rsi(df, n=14)
df = macd(df, fast=12, slow=26, signal=9)
df.write.partitionBy("year", "month").mode("overwrite").format("parquet").save("s3://dodgy-consulting-trading-bucket-gold/DOGEUSD/")

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 60edfe01-3d41-49c7-a53d-4c7587a16683
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true


Following exception encountered while creating session: An error occurred (AlreadyExistsException) when calling the CreateSession operation: Session already created, sessionId=60edfe01-3d41-49c7-a53d-4c7587a16683 

Error message: Session already created, sessionId=60edfe01-3d41-49c7-a53d-4c7587a16683 

Traceback (most recent call last):
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/aws_glue_interactive_sessions_kernel/glue_kernel_utils/KernelGateway.py", line 104, in create_session
    response = self.glue_client.create_session(
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/client.py", line 602, in _api_call
    return self._make_api_call(operation_name, kwargs)
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/client.py", line 1078, in _make_api_call
    raise error_class(parsed_response,